In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

In [67]:
s = np.load('fluxes_202505fdt.npz')
flux_density_1 = s['flux_density']
variance_1 = s['variance']
lat_1 = s['lat'].astype(float)

dlat_1 = lat_1[1] - lat_1[0]
lat_1 += dlat_1 / 2

In [68]:
s = np.load('fluxes_202509fdt.npz')
flux_density_2 = s['flux_density']
variance_2 = s['variance']
lat_2 = s['lat'].astype(float)

dlat_2 = lat_2[1] - lat_2[0]
lat_2 += dlat_2 / 2

In [69]:
fig = plt.figure(figsize=(10,8))
plt.errorbar(lat_1, flux_density_1, yerr=np.sqrt(variance_1))
plt.errorbar(lat_2, flux_density_2, yerr=np.sqrt(variance_2))

plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [70]:
def diffuse(y, li, q):
    from scipy.linalg import solve_banded
    
    V = - q * np.append(li[1:-1],0)
    U = - q * np.append(0,li[1:-1])
    D = (li[:-1] + li[1:]) / 2 * (1 + q * 2) 
    B = y * (li[:-1] + li[1:]) / 2
    
    D[-1] = li[-2] / 2 * (1 + q * 2) 
    D[0] = li[1] / 2 * (1 + q * 2) 
    
    return solve_banded((1, 1), [U, D, V], B, True, True)


In [71]:
R = 695e8
u = 800
d = 2e12

dx = dlat_1 * np.pi / 180 * R
dt = 24 * 3600

x = np.arange(-90,90) + 0.5
xi = np.arange(-90,90.5)

vi = u * np.sin(2 * xi * np.pi / 180) * dt / dx
li = 2 * np.pi * R * np.cos(xi * np.pi / 180)


l = 2 * np.pi * R * np.cos(x * np.pi / 180)
y = flux_density_1.copy()

for _ in range(4 * 30):
    y = y * l
    F = np.where(vi > 0, vi * np.append(0, y), vi * np.append(y, 0))
    dy = F[1:] - F[:-1]
    y = y - dy
    y = y / l
    
    y = diffuse(y, li, d * dt / dx ** 2)


In [72]:
x = np.arange(-90,90) + 0.5

plt.figure(figsize=(10,8))
plt.plot(x, flux_density_1)
plt.plot(x, y)
plt.xlim(-90,90)
plt.ylim(-15,15)
plt.grid(True)
plt.tight_layout()

In [43]:
from scipy.linalg import solve_banded

?solve_banded